In [1]:
ls

README.md   cpp_wrappers/  images/   mapping/   requirements.txt   trainer/
Test.ipynb  datasets/      infer.py  models/    semantickitti.zip  utils/
cfg/        debug.py       kernels/  nautilus/  train.py


In [2]:
%tb

import importlib
import argparse
from omegaconf import OmegaConf
import os.path as osp
from datasets.inference_dataset import *
from datasets import *
import torch 

args = {'source': 'nuscenes', 'target': 'semantickitti', 'cluster_cfg': './cfg/clust_cfg/cluster_20.yaml', 'model_cfg': './cfg/model_cfg/kp_sk_infer.yaml', 'data_cfg_path': './cfg/data_cfg', 'subsample': 1, 'save_pred_path': '/root/main/3DLabelProp/results_3DLabelProp'}

cfg = OmegaConf.create(args)
cluster_cfg = OmegaConf.load(cfg.cluster_cfg)
model_cfg = OmegaConf.load(cfg.model_cfg)
cfg = OmegaConf.merge(cfg,cluster_cfg,model_cfg)
if __name__ == "__main__":
    #Get info relative to the set
    if cfg.source == "semantickitti":
        source_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti.yaml"))
        train_set = SemanticKITTI(source_data_cfg,'valid')
    elif cfg.source == "nuscenes":
        source_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"nuscenes.yaml"))
        train_set = nuScenes(source_data_cfg,'valid')
    else:
        raise  NameError('source dataset not supported')

    if cfg.target == "semantickitti":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti.yaml"))
        val_set = SemanticKITTI(target_data_cfg,'valid')
    elif cfg.target == "nuscenes":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"nuscenes.yaml"))
        val_set = nuScenes(target_data_cfg,'valid')
    elif cfg.target == "semanticposs":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semanticposs.yaml"))
        val_set = SemanticPOSS(target_data_cfg,'valid')
    elif cfg.target == "semantickitti-nuscenes":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti-nuscenes.yaml"))
        val_set = SemanticKITTI_Nuscenes(target_data_cfg,'valid')
    elif "pandaset" in cfg.target:
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,cfg.target+".yaml"))
        val_set = Pandaset(target_data_cfg,'valid')
    
    else:
        raise  NameError('target dataset not supported')

    #Get info relative to the model
    if cfg.architecture.model == "KPCONV":
        module = importlib.import_module('models.kpconv.kpconv')
        model_information = getattr(module, cfg.architecture.type)()
        model_information.num_classes = train_set.get_n_label()
        model_information.ignore_label = -1
        model_information.in_features_dim = model_cfg.architecture.n_features
        from models.kpconv_model import SemanticSegmentationModel
        module = importlib.import_module('models.kpconv.architecture')
        model_type = getattr(module, cfg.architecture.type)
        model = SemanticSegmentationModel(model_information,cfg,model_type)
    elif cfg.architecture.model == "SPVCNN":
        module = importlib.import_module('models.spvcnn.spvcnn')
        model_information = getattr(module, cfg.architecture.type)
        model_information.num_classes = train_set.get_n_label()
        model_information.ignore_label = -1
        model_information.in_features_dim = model_cfg.architecture.n_features
        from models.spvcnn_model import SemanticSegmentationSPVCNNModel
        model = SemanticSegmentationSPVCNNModel(model_information,cfg)
    else:
        raise  NameError('model not supported')

    valid_dataset = InferenceDataset(cfg,train_set,val_set,model,model_information)
    #try:
    #    ius, miu = valid_dataset.compute_results()
    #except:
    valid_dataset.compute_dataset()
    ius, miu = valid_dataset.compute_results() # The results are already there?
    print(ius)
    print(miu)

No traceback available to show.


Model ready
Sequence:  ['08']


Processing dataset semantickitti:   0%|                                                           | 0/1 [01:48<?, ?it/s]


KeyboardInterrupt: 